In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Menambahkan root directory ke sys.path agar bisa import dari folder src
# Sesuaikan jumlah '../' tergantung letak file notebook ini. 
# Jika notebook ada di notebooks/knn/, maka butuh '../../'
sys.path.append(os.path.abspath('../../'))

from src.data_preparation import DataPreprocessor

# Setup visualisasi
%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

In [ ]:
# Load data (Sesuaikan path dengan struktur folder)
# Contoh: jika data ada di src/knn/data/hotel_bookings.csv
file_path = '../../src/knn/data/hotel_bookings.csv' 

try:
    df = pd.read_csv(file_path)
    print(f"Dataset berhasil diat dengan ukuran: {df.shape}")
except FileNotFoundError:
    print("File tidak ditekan. Pastikan path file csv sudah benar.")

# Lihat 5 baris pertama
df.head()

In [ ]:
# Cek jumlah nilai yang hilang
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print("Fitur dengan Missing Values:")
print(missing_values)

# Cek jumlah baris duplikat
print(f"\nJumlah baris duplikat: {df.duplicated().sum()}")

# Cek distribusi target (is_canceled)
print("\nDistribusi Target (is_canceled):")
print(df['is_canceled'].value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Pisahkan Fitur (X) dan Target (y)
# Asumsi kolom target bernama 'is_canceled'
target_col = 'is_canceled'

# Drop kolom yang berpotensi Data Leakage (misal: reservation_status)
# Ka perlu mengecek dataset asli, kolom seperti 'reservation_status' atau 'reservation_status_date' HARUS di-drop
cols_to_drop = [target_col, 'reservation_status', 'reservation_status_date']
cols_to_drop = [c for c in cols_to_drop if c in df.columns] # Filter jika kolom tidak ada

X = df.drop(columns=cols_to_drop)
y = df[target_col]

# 2. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Ukuran X_train: {X_train.shape}")
print(f"Ukuran X_test: {X_test.shape}")

In [ ]:
# Pisahkan kolom berdasarkan tipe datanya
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Jumlah kolom numerik: {len(numerical_cols)}")
print(f"Jumlah kolom kategorikal: {len(categorical_cols)}")

# (Opsional) Tampilkan beberapa nama kolom
print(f"\nContoh kolom numerik: {numerical_cols[:5]}")
print(f"Contoh kolom kategorikal: {categorical_cols[:5]}")

In [ ]:
# 1. Inisialisasi Preprocessor
prep = DataPreprocessor(numerical_cols=numerical_cols, categorical_cols=categorical_cols)

print("--- Memproses Training Data ---")
# 2. Fit & Transform pada Training Data
X_train_clean = prep.clean_data(X_train, is_train=True)
X_train_trans = prep.transform_features(X_train_clean, is_train=True)

# (Opsional) Feature Selection: Misal ambil 30 fitur terbaik
# X_train_final = prep.feature_selection(X_train_trans, y_train, k_best=30, is_train=True)
X_train_final = X_train_trans # Gunakan ini jika tidak pakai feature selection

# 3. Handle Imbalanced Data (Hanya pada train data!)
X_train_resampled, y_train_resampled = prep.handle_imbalance(X_train_final, y_train)


print("\n--- Memproses Test Data ---")
# 4. Transform pada Test Data (is_train=False)
X_test_clean = prep.clean_data(X_test, is_train=False)
X_test_trans = prep.transform_features(X_test_clean, is_train=False)

# Jika pakai feature selection di atas, terapkan juga di test:
# X_test_final = prep.feature_selection(X_test_trans, y=None, is_train=False)
X_test_final = X_test_trans 

In [ ]:
import os

# Tentukan direktori output
output_dir = '../../src/knn/data/processed/'
os.makedirs(output_dir, exist_ok=True)

# Simpan ke CSV
X_train_resampled.to_csv(f'{output_dir}X_train_processed.csv', index=False)
y_train_resampled.to_csv(f'{output_dir}y_train_processed.csv', index=False)

X_test_final.to_csv(f'{output_dir}X_test_processed.csv', index=False)
y_test.to_csv(f'{output_dir}y_test_processed.csv', index=False)

print(f"Dataset hasil preprocessing berhasil disimpan di folder: {output_dir}")
print(f"Bentuk akhir X_train: {X_train_resampled.shape}")
print(f"Bentuk akhir X_test: {X_test_final.shape}")

In [ ]:
# Agar tidak ada missing values setelah preprocessing
print("Missing values di X_train:", X_train_resampled.isnull().sum().sum())
print("Missing values di X_test:", X_test_final.isnull().sum().sum())

# Agar jumlah kolom di train dan test sama
print(f"Jumlah kolom X_train: {X_train_resampled.shape[1]}")
print(f"Jumlah kolom X_test: {X_test_final.shape[1]}")

# Agar tidak ada nilai tak hingga
print("Ada nilai infinity di train?", np.isinf(X_train_resampled.select_dtypes(include=np.number)).values.any())
print("Ada nilai infinity di test?", np.isinf(X_test_final.select_dtypes(include=np.number)).values.any())